In [ ]:
:set -XRankNTypes
:set -XScopedTypeVariables
putStrLn "Setup done."

# 🎓 Базовый Haskell

Запускай каждую ячейку через `Shift+Enter` и читай комментарии.

**Содержание:**
1. Базовые типы
2. Функции и типы
3. Списки
4. Паттерн-матчинг
5. Рекурсия
6. map / filter / fold
7. Лямбды и Currying
8. Maybe
9. ADT (Алгебраические типы)
10. Typeclasses
11. where / let / guards
12. Functor / Applicative / Monad
13. IO
14. Связанные ноутбуки

---

**Набор ноутбуков (по убыванию сложности):**
- `Monads.ipynb` -- полный справочник по всем монадам
- `FoldableTraversable.ipynb` -- Foldable, Traversable, Witherable
- `AlgebrasCoalgebras.ipynb` -- F-алгебры, катаморфизмы, схемы рекурсии
- `Adjunctions.ipynb` -- сопряжения, curry/uncurry в категорном смысле
- `Comonads.ipynb` -- комонады, duplicate, extend
- `MonadTransformers.ipynb` -- стек трансформеров


---
## 🔺 Теория категорий и Haskell — общая карта

Теория категорий изучает структуры через объекты (типы) и морфизмы (функции) между ними.

| Теория категорий | Haskell |
|---|---|
| Категория | Тип-система Haskell (`Hask`) |
| Объект | Тип (`Int`, `String`, `Maybe a`) |
| Морфизм (стрелка) | Функция `a -> b` |
| Композиция | Оператор `.` |
| id-морфизм | `id :: a -> a` |
| Функтор | Typeclass `Functor` |
| Ест. преобразование | Полиморфная функция `f a -> g a` |
| Монада | Typeclass `Monad` |
| Произведение | Кортеж `(a, b)` |
| Копроизведение | `Either a b` |
| Нач. объект | `Void` |
| Терм. объект | `()` |

> **Hask**: объекты — типы, морфизмы — функции. Все программы на Haskell — работа с этой категорией.

---
## 1️⃣ Базовые типы и выражения

In [1]:
-- Числа
42          -- Int
3.14        -- Double
2 ^ 10      -- степень

42

3.14

1024

In [2]:
-- Булевы и сравнения
True && False
True || False
not True

Line 2: Evaluate
Found:
True && False
Why not:
FalseLine 3: Evaluate
Found:
True || False
Why not:
TrueLine 4: Evaluate
Found:
not True
Why not:
False

False

True

False

In [3]:
-- Строки и символы
'H'
"Hello, Haskell!"
length "Haskell"

'H'

"Hello, Haskell!"

7

In [4]:
-- Кортежи (Tuple)
let pair = (42, "answer")
fst pair
-- snd pair

42

---
## 2️⃣ Функции и объявление типов

Формат: `имяФункции :: ТипАрг1 -> ТипАрг2 -> ТипРезультата`

### 🔺 Категорный взгляд: функции как морфизмы

Морфизм `f : A → B` — стрелка из A в B. В Haskell `f :: a -> b` — это морфизм в **Hask**.

**Законы категории** автоматически выполняются:
- **Тождество**: `id . f == f` и `f . id == f`
- **Ассоциативность**: `(f . g) . h == f . (g . h)`

In [5]:
double :: Int -> Int
double x = x * 2

double 21

42

In [6]:
add :: Int -> Int -> Int
add x y = x + y

add 3 4

7

In [7]:
-- Полиморфная функция (работает с любым типом a)
identity :: a -> a
identity x = x

identity 99

99

In [8]:
-- Законы категории: id и (.) в Haskell
f :: Int -> String
f n = show (n * 2)

-- Закон тождества
(id . f) 21 == f 21

-- Ассоциативность композиции
let g = length
let h = even
((h . g) . f) 21 == (h . (g . f)) 21

True

True

---
## 3️⃣ Списки

Список в Haskell — рекурсивная структура: либо пустой `[]`, либо элемент и хвост `x:xs`.

In [9]:
let nums  = [1, 2, 3, 4, 5]
let evens = [2, 4..20]   -- диапазон с шагом 2
evens

[2,4,6,8,10,12,14,16,18,20]

In [10]:
-- Основные функции
head [1,2,3]     -- 1
-- tail [1,2,3]  -- [2,3]
-- last [1,2,3]  -- 3
-- length [1,2,3]-- 3
-- reverse [1,2,3]-- [3,2,1]

1

In [11]:
-- cons (:) и конкатенация (++)
0 : [1, 2, 3]        -- [0,1,2,3]
-- [1,2] ++ [3,4]    -- [1,2,3,4]

[0,1,2,3]

In [12]:
-- List comprehension — как в математике: { x² | x ∈ [1..10], x чётный }
[x^2 | x <- [1..10], even x]

[4,16,36,64,100]

---
## 4️⃣ Паттерн-матчинг

Матчинг проверяет шаблоны сверху вниз — первое совпадение побеждает.

In [13]:
describe :: Int -> String
describe 0 = "zero"
describe 1 = "one"
describe n = "many: " ++ show n

describe 5

"many: 5"

In [14]:
-- Матчинг по структуре списка
myHead :: [a] -> a
myHead []    = error "empty list!"
myHead (x:_) = x   -- x — голова, _ — хвост (игнорируем)

myHead [10, 20, 30]

10

In [15]:
-- case выражение
classify :: Int -> String
classify n = case n `compare` 0 of
  LT -> "negative"
  EQ -> "zero"
  GT -> "positive"

classify (-5)
-- classify 0
-- classify 7

"negative"

---
## 5️⃣ Рекурсия

В Haskell нет цикло for/while — только рекурсия (и HOF поверх неё).

In [16]:
factorial :: Integer -> Integer
factorial 0 = 1
factorial n = n * factorial (n - 1)

factorial 10

3628800

In [17]:
fib :: Int -> Int
fib 0 = 0
fib 1 = 1
fib n = fib (n-1) + fib (n-2)

map fib [0..10]

[0,1,1,2,3,5,8,13,21,34,55]

In [18]:
myLength :: [a] -> Int
myLength []     = 0
myLength (_:xs) = 1 + myLength xs

myLength [1..100]

100

### 🔺 Категорный взгляд: рекурсия как fix-point и связь с F-алгебрами

Рекурсия в Haskell тесно связана с двумя категорными понятиями:

**1. fix :: (a -> a) -> a** -- оператор неподвижной точки
```haskell
fix :: (a -> a) -> a
fix f = let x = f x in x   -- x = f(f(f(f(...))))

-- Рекурсия через fix:
factorial = fix (\rec n -> if n == 0 then 1 else n * rec (n-1))
```

**2. Связь с F-алгебрами** -- каждый рекурсивный тип это **начальная алгебра** функтора

```
NatF a = ZeroF | SuccF a          -- функтор для натуральных чисел
Nat  = fix NatF = NatF (NatF (NatF ...))  -- начальная алгебра

ListF a b = NilF | ConsF a b      -- функтор для списков
[a] = fix (ListF a) = ListF a (ListF a ...)  -- начальная алгебра
```

**Катаморфизм** (`foldr`) -- единственная стрелка из начальной алгебры в любую другую:
```
[1,2,3]  --cata(alg)-->  alg(1, alg(2, alg(3, nil)))
sum      = cata (\case NilF -> 0; ConsF x acc -> x + acc)
product  = cata (\case NilF -> 1; ConsF x acc -> x * acc)
length   = cata (\case NilF -> 0; ConsF _ acc -> 1 + acc)
```

Так обобщаются `sum`, `product`, `length`, `map`, `filter`, `foldr` -- все это катаморфизмы.
Подробнее: `AlgebrasCoalgebras.ipynb`.


In [19]:
-- fix: fixed-point combinator
import Data.Function (fix)

-- Recursion via fix
let factFix :: (Int -> Int) -> Int -> Int
    factFix rec n = if n <= 0 then 1 else n * rec (n-1)
let factorial5 = fix factFix 5
putStrLn $ "fix factorial 5 = " ++ show factorial5

-- fib via fix
let fibFix :: (Int -> Int) -> Int -> Int
    fibFix rec n
        | n <= 0    = 0
        | n == 1    = 1
        | otherwise = rec (n-1) + rec (n-2)
let fib10 = fix fibFix 10
putStrLn $ "fix fib 10 = " ++ show fib10

-- Simulate fix with newtype to show structure
newtype Fix f = Fix { unFix :: f (Fix f) }

-- Natural numbers as Fix NatF
data NatF a = ZeroF | SuccF a deriving (Show, Functor)
type Nat = Fix NatF

let zero :: Nat
    zero = Fix ZeroF
let succ_ :: Nat -> Nat
    succ_ n = Fix (SuccF n)

-- cata: catamorphism (structural recursion = "fold")
let cata :: Functor f => (f a -> a) -> Fix f -> a
    cata alg = alg . fmap (cata alg) . unFix

-- toInt via cata
let toInt :: Nat -> Int
    toInt = cata (\x -> case x of { ZeroF -> 0; SuccF n -> n + 1 })

let two = succ_ (succ_ zero)
let three = succ_ two
putStrLn $ "toInt three = " ++ show (toInt three)
putStrLn $ "toInt (succ_ three) = " ++ show (toInt (succ_ three))

-- This is how foldr generalizes: it is cata for ListF
putStrLn "foldr f z = cata (NilF->z, ConsF x acc -> f x acc)"
let mySum = foldr (+) 0 [1..5]
putStrLn $ "sum [1..5] via foldr = " ++ show mySum


fix factorial 5 = 120

fix fib 10 = 55

toInt three = 3

toInt (succ_ three) = 4

foldr f z = cata (NilF->z, ConsF x acc -> f x acc)

sum [1..5] via foldr = 15

---
## 6️⃣ Higher-order функции: map, filter, fold

Функции первого класса — их можно передавать и возвращать как значения.

In [20]:
-- map f [a,b,c] == [f a, f b, f c]
map (*2) [1..5]

[2,4,6,8,10]

In [21]:
-- filter p xs == элементы xs, для которых p == True
filter even [1..10]

[2,4,6,8,10]

In [22]:
-- foldl f acc [a,b,c] == f(f(f(acc,a),b),c)
foldl (+) 0 [1..10]   -- сумма: 55
-- foldl (*) 1 [1..5] -- произведение: 120

Line 2: Use sum
Found:
foldl (+) 0
Why not:
sum

55

In [23]:
-- foldr f acc [a,b,c] == f a (f b (f c acc))
foldr (:) [] [1,2,3]   -- копия списка

[1,2,3]

---
## 7️⃣ Лямбды и Currying

В Haskell каждая функция принимает ровно 1 аргумент. `f x y` — это `(f x) y`.

### 🔺 Категорный взгляд: Currying и декартово замкнутые категории

**Декартово замкнутая категория (CCC)** — категория, где существует изоморфизм:

```
Hom(A x B, C)  ~=  Hom(A, C^B)
```

Тип `a -> b` в Haskell — это экспонент `B^A`.
- `curry :: ((a, b) -> c) -> a -> b -> c`
- `uncurry :: (a -> b -> c) -> (a, b) -> c`

In [24]:
-- Лямбда: \аргументы -> тело
map (\x -> x^2 + 1) [1..5]

[2,5,10,17,26]

In [25]:
-- Частичное применение: передаём меньше аргументов
add2 :: Int -> Int -> Int
add2 x y = x + y

let add5 = add2 5   -- новая функция Int -> Int
map add5 [1..5]     -- [6,7,8,9,10]

[6,7,8,9,10]

In [26]:
-- $ убирает скобки: f $ x == f x (низкий приоритет)
print $ map (*2) [1..5]

[2,4,6,8,10]

In [27]:
-- . — композиция функций: (f . g) x == f (g x)
let transform = filter even . map (*3)
transform [1..10]

[6,12,18,24,30]

In [28]:
-- curry/uncurry -- изоморфизм Hom(AxB,C) ~= Hom(A, C^B)
add_curried :: Int -> Int -> Int
add_curried x y = x + y

add_uncurried :: (Int, Int) -> Int
add_uncurried = uncurry add_curried

add_curried 3 4
-- add_uncurried (3, 4)  -- тот же результат

Line 2: Use camelCase
Found:
add_curried :: Int -> Int -> Int
Why not:
addCurried :: Int -> Int -> IntLine 3: Use camelCase
Found:
add_curried x y = ...
Why not:
addCurried x y = ...Line 5: Use camelCase
Found:
add_uncurried :: (Int, Int) -> Int
Why not:
addUncurried :: (Int, Int) -> IntLine 6: Use camelCase
Found:
add_uncurried = ...
Why not:
addUncurried = ...

7

---
## 8️⃣ Maybe — безопасные вычисления без исключений

`Maybe a = Nothing | Just a` — либо значения нет, либо оно есть.

In [29]:
safeDiv :: Int -> Int -> Maybe Int
safeDiv _ 0 = Nothing
safeDiv x y = Just (x `div` y)

safeDiv 10 2
-- safeDiv 10 0

Just 5

In [30]:
describeMaybe :: Maybe Int -> String
describeMaybe Nothing  = "нет значения"
describeMaybe (Just x) = "значение: " ++ show x

describeMaybe (safeDiv 10 0)

"\1085\1077\1090 \1079\1085\1072\1095\1077\1085\1080\1103"

In [31]:
-- maybe defaultValue function maybeValue
maybe 0 (*2) (Just 5)   -- 10
-- maybe 0 (*2) Nothing -- 0

10

---
## 9️⃣ Алгебраические типы данных (ADT)

`data` позволяет создавать свои типы. Sum type = «или», Product type = «и».

### 🔺 Категорный взгляд: произведения и копроизведения

**Произведение** (product) AxB — объект с проекциями:
- `fst :: (a, b) -> a`
- `snd :: (a, b) -> b`

**Копроизведение** (coproduct) A+B — объект с инъекциями:
- `Left  :: a -> Either a b`
- `Right :: b -> Either a b`

> `data Shape = Circle Double | Rectangle Double Double` — **тип-сумма** (копроизведение)
> `data Person = Person String Int` — **тип-произведение**

Любой ADT = сумма произведений типов.

In [32]:
data Shape = Circle Double
           | Rectangle Double Double
           deriving (Show)

area :: Shape -> Double
area (Circle r)      = pi * r^2
area (Rectangle w h) = w * h

area (Circle 5)
-- area (Rectangle 3 4)

78.53981633974483

In [33]:
-- Record syntax — именованные поля
data Person = Person
  { name :: String
  , age  :: Int
  } deriving (Show)

let alice = Person { name = "Alice", age = 30 }
name alice
-- age alice

"Alice"

In [34]:
-- Рекурсивный тип: бинарное дерево
data Tree a = Leaf | Node (Tree a) a (Tree a) deriving (Show)

let t = Node (Node Leaf 1 Leaf) 2 (Node Leaf 3 Leaf)
t

Node (Node Leaf 1 Leaf) 2 (Node Leaf 3 Leaf)

In [35]:
-- Универсальное свойство произведения: fanout
fanout :: (c -> a) -> (c -> b) -> c -> (a, b)
fanout f g x = (f x, g x)

-- Универсальное свойство копроизведения: fanin (= either)
fanin :: (a -> c) -> (b -> c) -> Either a b -> c
fanin f _ (Left x)  = f x
fanin _ g (Right y) = g y

fanin (+1) (*10) (Left 5)
-- fanin (+1) (*10) (Right 5)

6

---
## 🔟 Классы типов (Typeclasses)

Typeclass = интерфейс/контракт. Тип «реализует» класс через `instance`.

### 🔺 Категорный взгляд: классы типов как алгебраические структуры

| Typeclass | Категорная структура |
|---|---|
| `Eq` | Отношение эквивалентности |
| `Ord` | Частичный порядок (poset) |
| `Semigroup` | Полугруппа |
| `Monoid` | Моноид |
| `Functor` | Эндофунктор Hask → Hask |
| `Foldable` | Уничтожение структуры в моноид |
| `Traversable` | Коммутирование функторов |

**Monoid** — категория с одним объектом: морфизмы — элементы, композиция — `(<>)`, id — `mempty`.

In [36]:
-- Свой класс типов
class Describable a where
  describe' :: a -> String

data Color = Red | Green | Blue deriving (Show)

instance Describable Color where
  describe' Red   = "горячий красный"
  describe' Green = "спокойный зелёный"
  describe' Blue  = "холодный синий"

describe' Green

"\1089\1087\1086\1082\1086\1081\1085\1099\1081 \1079\1077\1083\1105\1085\1099\1081"

In [37]:
-- deriving автоматически реализует стандартные классы
data Priority = Low | Medium | High deriving (Show, Eq, Ord, Enum)

High > Low
-- [Low .. High]   -- [Low,Medium,High]

True

In [38]:
-- Ограничение класса в сигнатуре: Ord a =>
largest :: Ord a => [a] -> a
largest [x]    = x
largest (x:xs) = max x (largest xs)

largest [3, 1, 4, 1, 5, 9, 2, 6]
-- largest "haskell"  -- работает и для строк!

9

In [39]:
-- Monoid = категория с одним объектом
-- mempty = id, (<>) = композиция

-- Законы моноида (= законы категории):
-- mempty <> x == x
-- x <> mempty == x
-- (x <> y) <> z == x <> (y <> z)

import Data.Monoid (Sum(..), Product(..))

getSum     (foldMap Sum     [1..5])  -- 15 (моноид (+, 0))
-- getProduct (foldMap Product [1..5])  -- 120 (моноид (*, 1))
-- mconcat [[1,2],[3,4],[5]] -- [1,2,3,4,5]  (моноид списков)

15

---
## 1️⃣1️⃣ Where, Let и Guards

In [40]:
-- Guards (|) + where
bmi :: Double -> String
bmi weight
  | val < 18.5 = "недовес"
  | val < 25.0 = "норма"
  | val < 30.0 = "лишний вес"
  | otherwise  = "ожирение"
  where
    val = weight / 1.75^2

bmi 70
-- bmi 50
-- bmi 100

"\1085\1086\1088\1084\1072"

In [41]:
-- let ... in
let result = let x = 10
                 y = 20
             in x + y
result

30

---
## 1️⃣2️⃣ Functor / Applicative / Monad

Это «интерфейсы» для работы с контейнерами/контекстами обобщённым способом.

### 🔺 Категорный взгляд: Functor, Applicative, Monad

#### Функтор (Functor)
**Функтор** `F : C → D` — отображение между категориями:
- `a ↦ F a` (объекты)
- `(a->b) ↦ (F a -> F b)` — это `fmap`
- `fmap id == id`
- `fmap (f . g) == fmap f . fmap g`

В Haskell `Functor` — **эндофунктор** Hask → Hask.

#### Естественное преобразование
`α : F ⟹ G` — семейство морфизмов `α_A : F(A) → G(A)`.
В Haskell — полиморфная функция `f a -> g a` (parametricity гарантирует натуральность!).

#### Монада
**Монада** — моноид в категории эндофункторов `[Hask,Hask]`:
- Эндофунктор `T`
- `return` = `η : Id ⟹ T` (единица)
- `join` = `μ : T² ⟹ T` (умножение)
- `m >>= f = join (fmap f m)`

In [42]:
-- Functor: fmap :: (a -> b) -> f a -> f b
-- «применить функцию внутри контейнера»
fmap (*2) (Just 5)    -- Just 10
-- fmap (*2) Nothing  -- Nothing
-- fmap (*2) [1..5]   -- [2,4,6,8,10]

Just 10

In [43]:
-- Законы функтора для Maybe

-- Закон 1: fmap id == id
fmap id (Just 42) == id (Just 42)

-- Закон 2: fmap (f . g) == fmap f . fmap g
let f = (+1) :: Int -> Int
let g = (*2) :: Int -> Int
fmap (f . g) (Just 5) == (fmap f . fmap g) (Just 5)

True

True

In [44]:
-- Applicative: <*> :: f (a -> b) -> f a -> f b
-- «функция тоже в контейнере»
Just (*3) <*> Just 7
-- [(+1),(*2)] <*> [10,20]  -- [11,21,20,40]

Just 21

In [45]:
-- Естественное преобразование: alpha : F => G
-- Натуральность: fmap f . alpha == alpha . fmap f

-- alpha: Maybe => []
maybeToList :: Maybe a -> [a]
maybeToList Nothing  = []
maybeToList (Just x) = [x]

-- Проверка натуральности:
fmap (+1) (maybeToList (Just 5)) == maybeToList (fmap (+1) (Just 5))

True

In [46]:
-- Monad: >>= :: m a -> (a -> m b) -> m b
-- в IHaskell многострочный >>= недоступен, используйте скобки или do
Just 5 >>= (\x -> Just (x * 2) >>= (\y -> Just (y + 1)))

Just 11

In [47]:
-- Монада через join (mu : T^2 => T)

-- join для Maybe:
joinMaybe :: Maybe (Maybe a) -> Maybe a
joinMaybe Nothing         = Nothing
joinMaybe (Just Nothing)  = Nothing
joinMaybe (Just (Just x)) = Just x

-- bind через join: m >>= f = join (fmap f m)
myBind :: Maybe a -> (a -> Maybe b) -> Maybe b
myBind m f = joinMaybe (fmap f m)

myBind (Just 5) (\x -> Just (x * 2))
-- join для списков = concat:
-- concat [[1,2],[3,4],[5]]

Just 10

In [48]:
-- do-нотация — синтаксический сахар для >>= 
result :: Maybe Int
result = do
  x <- Just 5
  y <- Just 10
  return (x + y)

result

Just 15

In [49]:
-- List Monad: недетерминированные вычисления
pairs :: [(Int, Int)]
pairs = do
  x <- [1..3]
  y <- [1..3]
  return (x, y)

pairs

[(1,1),(1,2),(1,3),(2,1),(2,2),(2,3),(3,1),(3,2),(3,3)]

In [50]:
-- Законы монады = законы моноида в [Hask,Hask]

-- Закон 1: return x >>= f == f x
(return 5 >>= \x -> Just (x*2)) == Just (5*2)

-- Закон 2: m >>= return == m
(Just 5 >>= return) == Just 5

-- Закон 3: ассоциативность
let m = Just 5
let f x = Just (x + 1)
let g x = Just (x * 2)
((m >>= f) >>= g) == (m >>= (\x -> f x >>= g))

True

True

True

---
## 1️⃣3️⃣ IO Монада

Всё, что взаимодействует с внешним миром — ввод/вывод, файлы — живёт в `IO`.
Это делает побочные эффекты **явными** в системе типов.

In [51]:
putStrLn "Hello from Haskell IO!"

Hello from Haskell IO!

In [52]:
main :: IO ()
main = do
  let nums = [1..5]
  putStrLn $ "Список: "  ++ show nums
  putStrLn $ "Сумма: "   ++ show (sum nums)
  putStrLn $ "Среднее: " ++ show (fromIntegral (sum nums) / fromIntegral (length nums) :: Double)

main

Список: [1,2,3,4,5]
Сумма: 15
Среднее: 3.0

---
## 🏁 Итог тура

| Концепция | Ключевые слова |
|-----------|----------------|
| Типы | `::`, `->`, `=>` |
| Паттерны | `case`, `of`, `_` |
| Данные | `data`, `deriving` |
| Классы типов | `class`, `instance` |
| Монады | `do`, `>>=`, `return` |
| Локальные имена | `where`, `let` |
| Ветвление | guards `|`, `otherwise` |
| HOF | `map`, `filter`, `fold` |

**Следующие шаги:**
- 📖 [Learn You a Haskell](http://learnyouahaskell.com) — бесплатная книга для начинающих
- 📖 [Real World Haskell](http://book.realworldhaskell.org) — практика
- 🔥 [Haskell Exercises on Exercism](https://exercism.org/tracks/haskell) — практические задачи


---
**Следующий →** [TypeAlgebra](TypeAlgebra.ipynb)